# Week 7: The Adaptation Decision and a Production Dataset

In [13]:
%pip install -q tiktoken
%pip install -q sentence-transformers
%pip install -q openai
%pip install -q python-dotenv

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## Part 1: Load the Home-Care Request Dataset

In [14]:
import json
from pathlib import Path
from collections import Counter

def load_jsonl(filename):
    return [
        json.loads(line)
        for line in Path(filename).read_text().splitlines()
        if line.strip()
    ]

train = load_jsonl("test-data/train.jsonl")
eval_ = load_jsonl("test-data/eval.jsonl")
VALID = {record["label"] for record in train}

print("Training examples:", len(train))
print("Evaluation examples:", len(eval_))
print("Training labels:", Counter(r["label"] for r in train))
print("First training example:", train[0])

Training examples: 200
Evaluation examples: 50
Training labels: Counter({'scheduling': 40, 'billing': 40, 'document_request': 40, 'visit_verification': 40, 'caregiver_assignment': 40})
First training example: {'text': 'I want to reduce the weekly schedule from five days to three.', 'label': 'scheduling'}


### Clean, Validate, and Report Dataset Quality

In [22]:
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import tiktoken

ALLOWED_LABELS = {
    "scheduling",
    "billing",
    "visit_verification",
    "caregiver_assignment",
    "document_request",
}

def normalize(text):
    return re.sub(r"\s+", " ", text.strip()).casefold()


def clean_records(records):
    cleaned = []
    seen = {}
    removed = Counter()
    conflicts = []

    for record in records:
        # Filtering and format validation
        if not isinstance(record, dict) or set(record) != {"text", "label"}:
            removed["invalid_fields"] += 1
            continue

        text, label = record["text"], record["label"]

        if not isinstance(text, str) or not isinstance(label, str):
            removed["invalid_types"] += 1
            continue

        text = re.sub(r"\s+", " ", text.strip())
        label = label.strip()

        if not text:
            removed["blank_text"] += 1
            continue

        if label not in ALLOWED_LABELS:
            removed["invalid_label"] += 1
            continue

        # Deduplicate by input, regardless of capitalization/whitespace
        key = normalize(text)

        if key in seen:
            if seen[key] != label:
                conflicts.append({
                    "text": text,
                    "labels": [seen[key], label],
                })
            else:
                removed["duplicate_input"] += 1
            continue

        seen[key] = label
        cleaned.append({"text": text, "label": label})

    return cleaned, {
        "raw_count": len(records),
        "clean_count": len(cleaned),
        "removed_by_reason": dict(removed),
        "conflicting_labels": conflicts,
    }


# Reload original files so rerunning this cell preserves raw counts.
def read_records(filename):
    return [
        json.loads(line)
        for line in Path(filename).read_text().splitlines()
        if line.strip()
    ]


train, train_report = clean_records(read_records("test-data/train.jsonl"))
eval_, eval_report = clean_records(read_records("test-data/eval.jsonl"))

# Detect leakage; keep evaluation inputs and remove matching train inputs.
eval_inputs = {normalize(r["text"]) for r in eval_}
overlap_count = sum(normalize(r["text"]) in eval_inputs for r in train)

train = [
    r for r in train
    if normalize(r["text"]) not in eval_inputs
]

train_report["removed_train_eval_overlap"] = overlap_count
train_report["clean_count"] = len(train)

# Token counts are a proxy, not Gemini's actual tokenizer.
enc = tiktoken.get_encoding("cl100k_base")

def dataset_statistics(records):
    lengths = [
        len(enc.encode(r["text"], disallowed_special=()))
        for r in records
    ]

    return {
        "label_balance": dict(Counter(r["label"] for r in records)),
        "token_lengths": {
            "min": min(lengths),
            "mean": round(float(np.mean(lengths)), 2),
            "median": float(np.median(lengths)),
            "p95": round(float(np.percentile(lengths, 95)), 2),
            "max": max(lengths),
            "distribution": {
                "1_to_10": sum(n <= 10 for n in lengths),
                "11_to_20": sum(10 < n <= 20 for n in lengths),
                "21_to_30": sum(20 < n <= 30 for n in lengths),
                "31_plus": sum(n > 30 for n in lengths),
            },
        },
        "format_consistent": all(
            set(r) == {"text", "label"}
            and isinstance(r["text"], str)
            and bool(r["text"].strip())
            and r["label"] in ALLOWED_LABELS
            for r in records
        ),
    }


quality_report = {
    "source": "AI-generated synthetic home-care requests.",
    "human_label_review": "Completed",
    "filtering": (
        "Remove invalid fields/types, blank text, and unsupported labels; "
        "normalize whitespace."
    ),
    "deduplication": (
        "Compare inputs after case and whitespace normalization. "
        "Semantic near-duplicates require separate review."
    ),
    "tokenizer": "cl100k_base; proxy for Gemini token lengths",
    "training": train_report,
    "evaluation": eval_report,
    "validation": {
        "minimum_train_size_met": len(train) >= 200,
        "minimum_eval_size_met": len(eval_) >= 50,
        "remaining_train_eval_overlap": len(
            {normalize(r["text"]) for r in train} & eval_inputs
        ),
        "no_conflicting_labels": not (
            train_report["conflicting_labels"]
            or eval_report["conflicting_labels"]
        ),
    },
}

# Save diagnostics even if validation fails.
Path("quality_report.json").write_text(
    json.dumps(quality_report, indent=2),
    encoding="utf-8",
)

assert quality_report["validation"]["no_conflicting_labels"], (
    "Resolve conflicting labels in the original files, then rerun."
)
assert len(train) >= 200, "Add training records after cleaning to reach 200."
assert len(eval_) >= 50, "Add evaluation records after cleaning to reach 50."
assert quality_report["validation"]["remaining_train_eval_overlap"] == 0

quality_report["training"].update(dataset_statistics(train))
quality_report["evaluation"].update(dataset_statistics(eval_))

Path("quality_report.json").write_text(
    json.dumps(quality_report, indent=2),
    encoding="utf-8",
)

VALID = ALLOWED_LABELS

print(json.dumps(quality_report, indent=2))
print("\nSaved quality_report.json")

{
  "source": "AI-generated synthetic home-care requests.",
  "human_label_review": "Completed",
  "filtering": "Remove invalid fields/types, blank text, and unsupported labels; normalize whitespace.",
  "deduplication": "Compare inputs after case and whitespace normalization. Semantic near-duplicates require separate review.",
  "tokenizer": "cl100k_base; proxy for Gemini token lengths",
  "training": {
    "raw_count": 200,
    "clean_count": 200,
    "removed_by_reason": {},
    "conflicting_labels": [],
    "removed_train_eval_overlap": 0,
    "label_balance": {
      "scheduling": 40,
      "billing": 40,
      "document_request": 40,
      "visit_verification": 40,
      "caregiver_assignment": 40
    },
    "token_lengths": {
      "min": 7,
      "mean": 11.46,
      "median": 11.0,
      "p95": 15.05,
      "max": 18,
      "distribution": {
        "1_to_10": 72,
        "11_to_20": 128,
        "21_to_30": 0,
        "31_plus": 0
      }
    },
    "format_consistent": true


## Part 2: Retrieve Relevant Few-Shot Examples

In [16]:
from sentence_transformers import SentenceTransformer, util

# Load an embedding model and encode the 200 training inputs.
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

train_emb = emb.encode(
    [record["text"] for record in train],
    convert_to_tensor=True,
    normalize_embeddings=True,
)

def few_shot(query, k=4):
    query_emb = emb.encode(
        [query],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    similarities = util.cos_sim(query_emb, train_emb)[0]
    indices = similarities.topk(k).indices.tolist()

    return [train[i] for i in indices]


# Demonstrate retrieval for one evaluation question.
example = eval_[0]

print("Evaluation question:", example["text"])
print("Correct answer:", example["label"])
print("\nRetrieved examples:")

for record in few_shot(example["text"]):
    print(f'{record["text"]} → {record["label"]}')

Evaluation question: Before deciding about a replacement, I'd like the blank reassignment paperwork.
Correct answer: document_request

Retrieved examples:
I need a copy of the renewal agreement before I sign it. → document_request
I'd like the procedure for requesting a new caregiver as a written handout. → document_request
I'd like a written copy of the cancellation policy. → document_request
I need a duplicate of the form I signed during intake. → document_request


In [17]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(".env")

api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY is missing from .env")

client = OpenAI(
    api_key=api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
MODEL = "gemini-3.5-flash-lite"

INSTRUCTIONS = """
Classify the home-care request into exactly one label:

scheduling: Arrange or change future visit times, duration, or frequency.
billing: Handle charges, payments, refunds, rates, or account balances.
visit_verification: Fix or verify actual attendance and electronic visit records.
caregiver_assignment: Select, replace, retain, or identify an assigned caregiver.
document_request: Obtain documents, copies, reports, instructions, or blank forms.

Classify by the main requested action, not background topics.
Return only the label, without explanation.
""".strip()


def classify(text, shots=None):
    prompt = INSTRUCTIONS + "\n\n"

    if shots:
        prompt += "Examples:\n"
        for record in shots:
            prompt += (
                f'Request: {record["text"]}\n'
                f'Label: {record["label"]}\n\n'
            )

    prompt += f"Request: {text}\nLabel:"

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )

    return (response.choices[0].message.content or "").strip().lower()

## Part 3: Evaluate Zero-Shot and Dynamic Few-Shot Performance

In [18]:
import json
from pathlib import Path
import time

results = []

for i, record in enumerate(eval_, start=1):
    shots = few_shot(record["text"])


    zero_prediction = classify(record["text"])
    time.sleep(5)

    few_prediction = classify(record["text"], shots=shots)
    time.sleep(5)

    results.append({
        "text": record["text"],
        "correct_label": record["label"],
        "zero_shot_prediction": zero_prediction,
        "few_shot_prediction": few_prediction,
        "zero_shot_correct": zero_prediction == record["label"],
        "few_shot_correct": few_prediction == record["label"],
        "retrieved_examples": shots,
    })

    # Save progress after each completed question.
    Path("evaluation_results.json").write_text(
        json.dumps(results, indent=2),
        encoding="utf-8",
    )

    print(f"Completed {i}/{len(eval_)}")


zero_accuracy = sum(r["zero_shot_correct"] for r in results) / len(results)
few_accuracy = sum(r["few_shot_correct"] for r in results) / len(results)

print(f"\nZero-shot accuracy: {zero_accuracy:.1%}")
print(f"Dynamic few-shot accuracy: {few_accuracy:.1%}")
print(f"Difference: {(few_accuracy - zero_accuracy) * 100:+.1f} percentage points")

Completed 1/50
Completed 2/50
Completed 3/50
Completed 4/50
Completed 5/50
Completed 6/50
Completed 7/50
Completed 8/50
Completed 9/50
Completed 10/50
Completed 11/50
Completed 12/50
Completed 13/50
Completed 14/50
Completed 15/50
Completed 16/50
Completed 17/50
Completed 18/50
Completed 19/50
Completed 20/50
Completed 21/50
Completed 22/50
Completed 23/50
Completed 24/50
Completed 25/50
Completed 26/50
Completed 27/50
Completed 28/50
Completed 29/50
Completed 30/50
Completed 31/50
Completed 32/50
Completed 33/50
Completed 34/50
Completed 35/50
Completed 36/50
Completed 37/50
Completed 38/50
Completed 39/50
Completed 40/50
Completed 41/50
Completed 42/50
Completed 43/50
Completed 44/50
Completed 45/50
Completed 46/50
Completed 47/50
Completed 48/50
Completed 49/50
Completed 50/50

Zero-shot accuracy: 94.0%
Dynamic few-shot accuracy: 98.0%
Difference: +4.0 percentage points


### Identify Questions Where Dynamic Few-Shot Performed Worse

Find requests that zero-shot classified correctly but dynamic few-shot classified incorrectly. Display their predictions and retrieved examples for inspection.

In [19]:
regressions = [
    r for r in results
    if r["zero_shot_correct"] and not r["few_shot_correct"]
]

print("Questions where few-shot did worse:", len(regressions))

for r in regressions:
    print("\nQuestion:", r["text"])
    print("Correct label:", r["correct_label"])
    print("Zero-shot:", r["zero_shot_prediction"])
    print("Few-shot:", r["few_shot_prediction"])

    print("Retrieved examples:")
    for example in r["retrieved_examples"]:
        print(f'  {example["text"]} → {example["label"]}')

Questions where few-shot did worse: 1

Question: We kept every appointment, yet there's a no-show penalty on the statement. Please correct it.
Correct label: billing
Zero-shot: billing
Few-shot: visit_verification
Retrieved examples:
  Don't move the next appointment; update the attendance record for today's visit. → visit_verification
  The statement includes a visit we cancelled; remove that charge. → billing
  Please remove the extra appointment booked for Sunday. → scheduling
  The scheduled visits are right; it's the amount due that's wrong. → billing


In [20]:
for label in sorted(VALID):
    subset = [r for r in results if r["correct_label"] == label]

    zero = sum(r["zero_shot_correct"] for r in subset) / len(subset)
    few = sum(r["few_shot_correct"] for r in subset) / len(subset)

    print(
        f"{label}: n={len(subset)}, "
        f"zero-shot={zero:.0%}, few-shot={few:.0%}"
    )

billing: n=10, zero-shot=100%, few-shot=90%
caregiver_assignment: n=10, zero-shot=100%, few-shot=100%
document_request: n=10, zero-shot=70%, few-shot=100%
scheduling: n=10, zero-shot=100%, few-shot=100%
visit_verification: n=10, zero-shot=100%, few-shot=100%


### Part 3 Analysis: Compare Accuracy by Label Category

Overall, zero-shot achieved 94% accuracy (47/50), and dynamic few-shot
achieved 98% accuracy (49/50). Few-shot missed one billing request, while
zero-shot missed three document requests. Few-shot was slightly more accurate
overall, answering two additional questions correctly.

However, on the billing evaluation slice (n=10), zero-shot achieved 100%
accuracy, while dynamic few-shot achieved 90%.

The failure where few-shot performed worse was: “We kept every appointment,
yet there's a no-show penalty on the statement. Please correct it.”
The correct label was `billing`, but few-shot predicted `visit_verification`.

This mislabeling may have been influenced by the question mentioning
appointments and specifics around it. The retrieved examples included a
`visit_verification` example that also mentioned the keyword appointment. This
similarity could have contributed to the misclassification, even though
the original request asks to correct a financial penalty rather than an
attendance record.

## Part 4: Decision Memo

I created a scenario where a home-care company receives requests covering five categories: scheduling, billing, visit_verification, caregiver_assignment, and document_request. Dynamic few-shot prompting achieved 98% accuracy (49/50), with one misclassification in the billing evaluation slice. Zero-shot prompting achieved 94% accuracy (47/50), with three misclassifications in the document_request slice. These results are based on 50 AI-generated test requests.

Clearer prompt instructions could improve classification by explaining the difference between correcting a billing charge and correcting an attendance record. This could address the confusion around appointments seen in the billing failure, although that prompt change would need to be tested. Retrieval added implementation complexity and processing time to the dynamic few-shot approach. Together, retrieval and few-shot prompting improved accuracy from 94% to 98%, only two additional correct answers out of 50. This small improvement may not justify the added complexity for this task.

Although dynamic few-shot performed slightly better, the improvement was marginal and required more prompt tokens and a retrieval step. Keeping the existing zero-shot approach without further adaptation could be reasonable if 94% accuracy meets the business needs and errors receive human review. My next step would be to refine the instructions and evaluate a larger, more realistic dataset.

Based on the accuracy achieved with both zero-shot and dynamic few-shot prompting, fine-tuning does not appear justified at this stage. I would consider it if accuracy remained below an acceptable level—for example, 80% or less—despite refining the prompts and example selection. This would also require more reviewed training data and evidence that the improvement would justify the training and maintenance costs.